# Machine Learning Based Intrusion Detection System

A compact Colab demonstration of a network intrusion detection pipeline.

**Flow:** Dataset → Cleaning → Sampling → Feature/Target split → Train/Test split → ML models → Evaluation → Prediction

In [ ]:
# 1. Install / import libraries
!pip -q install xgboost

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
from xgboost import XGBClassifier

## 1. Load the network traffic dataset

We use a sampled CICIDS2017 CSV so the complete experiment can run during a practical demonstration.

Each row represents a network flow and the columns contain numerical traffic features. The `Label` column tells us whether the flow is benign or belongs to an attack category.

In [ ]:
# 2. Load sampled CICIDS2017 data
DATA_URL = "https://raw.githubusercontent.com/Western-OC2-Lab/Intrusion-Detection-System-Using-Machine-Learning/main/data/CICIDS2017_sample.csv"

df = pd.read_csv(DATA_URL)

print("Dataset shape:", df.shape)
display(df.head())

In [ ]:
# 3. Inspect labels and basic data quality
print("Labels:")
display(df["Label"].value_counts())

print("\nMissing values:", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

## 2. Prepare the data

For this demonstration we convert the original attack categories into two practical IDS classes:

- `BENIGN` → normal traffic
- Everything else → `ATTACK`

This makes the final prediction easy to understand during a live demonstration.

In [ ]:
# 4. Clean and prepare the data

df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna()

# Remove exact duplicate traffic records
df = df.drop_duplicates()

# Convert the original multi-class labels to a binary IDS target
df["Target"] = (df["Label"].str.upper() != "BENIGN").astype(int)

print(df["Target"].value_counts().rename(index={0: "BENIGN", 1: "ATTACK"}))

In [ ]:
# 5. Visualize the class distribution

counts = df["Target"].value_counts().sort_index()
plt.figure(figsize=(6, 4))
plt.bar(["BENIGN", "ATTACK"], counts.values)
plt.title("Network Traffic Distribution")
plt.ylabel("Number of flows")
plt.show()

## 3. Create a manageable training set

The full network dataset can be large. For a live demonstration, we use a stratified sample so both classes are represented while keeping training time reasonable.

In [ ]:
# 6. Stratified sample for a fast classroom demo

MAX_ROWS = 30000

if len(df) > MAX_ROWS:
    demo_df, _ = train_test_split(
        df,
        train_size=MAX_ROWS,
        stratify=df["Target"],
        random_state=42
    )
else:
    demo_df = df.copy()

X = demo_df.drop(columns=["Label", "Target"])
y = demo_df["Target"]

# Make sure all feature columns are numeric
X = X.apply(pd.to_numeric, errors="coerce")
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

print("Rows used:", len(X))
print("Features used:", X.shape[1])

In [ ]:
# 7. Train/test split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows :", len(X_test))

## 4. Train the intrusion detection models

We compare four tree-based classifiers:

- Decision Tree
- Random Forest
- Extra Trees
- XGBoost

Tree models are useful here because the dataset contains many numerical network-flow features and the models can learn non-linear relationships between those features and the traffic class.

In [ ]:
# 8. Define models

models = {
    "Decision Tree": DecisionTreeClassifier(
        max_depth=20,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=120,
        random_state=42,
        n_jobs=-1
    ),

    "Extra Trees": ExtraTreesClassifier(
        n_estimators=120,
        random_state=42,
        n_jobs=-1
    ),

    "XGBoost": XGBClassifier(
        n_estimators=120,
        max_depth=8,
        learning_rate=0.1,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )
}

In [ ]:
# 9. Train and evaluate all models

results = []
trained_models = {}

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)

    pred = model.predict(X_test)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1 Score": f1_score(y_test, pred, zero_division=0)
    })

    trained_models[name] = model

results_df = pd.DataFrame(results).sort_values("F1 Score", ascending=False)
display(results_df.style.format({
    "Accuracy": "{:.2%}",
    "Precision": "{:.2%}",
    "Recall": "{:.2%}",
    "F1 Score": "{:.2%}"
}))

In [ ]:
# 10. Compare model performance

plot_df = results_df.set_index("Model")[["Accuracy", "Precision", "Recall", "F1 Score"]]

ax = plot_df.plot(kind="bar", figsize=(10, 5), ylim=(0, 1))
ax.set_title("Model Performance Comparison")
ax.set_ylabel("Score")
ax.set_xlabel("")
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 5. Inspect the best model

For an IDS, accuracy alone is not enough. We also check precision, recall, F1-score and the confusion matrix.

The confusion matrix shows:

- True Negative: benign traffic correctly identified
- False Positive: benign traffic flagged as an attack
- False Negative: an attack missed by the model
- True Positive: attack correctly detected

In [ ]:
# 11. Detailed evaluation of the best model

best_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_name]

best_pred = best_model.predict(X_test)

print("Selected model:", best_name)
print()
print(classification_report(
    y_test,
    best_pred,
    target_names=["BENIGN", "ATTACK"],
    digits=4
))

In [ ]:
# 12. Confusion matrix

cm = confusion_matrix(y_test, best_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["BENIGN", "ATTACK"]
)

disp.plot()
plt.title(f"Confusion Matrix - {best_name}")
plt.show()

## 6. Feature importance

The model can also tell us which network-flow features contributed most to its decisions.

This is useful because it gives us an idea of what characteristics of the traffic are important for detecting attacks.

In [ ]:
# 13. Show the most important network features

if hasattr(best_model, "feature_importances_"):
    importance = pd.Series(
        best_model.feature_importances_,
        index=X.columns
    ).sort_values(ascending=False).head(15)

    plt.figure(figsize=(9, 6))
    importance.sort_values().plot(kind="barh")
    plt.title(f"Top Network Features - {best_name}")
    plt.xlabel("Importance")
    plt.tight_layout()
    plt.show()

    display(importance.to_frame("Importance"))
else:
    print("Feature importance is not available for this model.")

## 7. Live prediction

Now we use the trained model on traffic records that were not used during training.

The output is converted into a simple IDS decision: `BENIGN` or `ATTACK`.

In [ ]:
# 14. Predict a few unseen traffic records

sample = X_test.iloc[:10]
predictions = best_model.predict(sample)

prediction_table = pd.DataFrame({
    "Actual": np.where(y_test.iloc[:10].values == 1, "ATTACK", "BENIGN"),
    "Predicted": np.where(predictions == 1, "ATTACK", "BENIGN")
})

display(prediction_table)

In [ ]:
# 15. Simple reusable prediction function

def detect_intrusion(model, traffic_rows):
    traffic_rows = traffic_rows.copy()
    traffic_rows = traffic_rows.reindex(columns=X.columns, fill_value=0)
    traffic_rows = traffic_rows.apply(pd.to_numeric, errors="coerce")
    traffic_rows = traffic_rows.replace([np.inf, -np.inf], np.nan).fillna(0)

    predictions = model.predict(traffic_rows)

    return np.where(predictions == 1, "ATTACK", "BENIGN")


# Example: classify one unseen flow
result = detect_intrusion(best_model, X_test.iloc[[0]])
print("IDS prediction:", result[0])

## Final pipeline

**Network traffic → Data cleaning → Sampling → Feature/target preparation → Train/test split → ML model → Prediction → IDS decision**

The important limitation is that this is a dataset-based machine-learning prototype. A real deployment would also need continuous traffic capture, feature extraction from live packets/flows, monitoring, retraining, and handling of previously unseen attack patterns.